# PhoBERT Multi-Label Moderation — Training Notebook

Fine-tunes `vinai/phobert-base` for **3-label** multi-label classification:
**toxic | spam | hate_speech**

> nsfw và off_topic bị loại vì không có training data.

## Setup
- **Google Colab**: Runtime → Change runtime type → **T4 GPU**
- **Kaggle**: Notebook Settings → Accelerator → GPU P100

Sau khi train xong model được push lên HuggingFace Hub để AI service load về.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install -q transformers datasets accelerate wandb scikit-learn huggingface_hub

In [ ]:
# ── Cell 2: Config — EDIT THESE ───────────────────────────────────────────
import os

HF_MODEL_REPO  = "your-hf-username/fshop-phobert-moderation"  # CHANGE
WANDB_API_KEY  = "your-wandb-api-key"                          # CHANGE
WANDB_PROJECT  = "fshop-moderation"
WANDB_RUN_NAME = "phobert-v1"

# Colab: upload unified_dataset.csv rồi để path mặc định
# Kaggle: đổi thành "/kaggle/input/fshop-moderation/unified_dataset.csv"
DATASET_PATH = "unified_dataset.csv"

MODEL_NAME  = "vinai/phobert-base"
LABEL_NAMES = ["toxic", "spam", "hate_speech"]   # 3 labels có data thực sự
MAX_LENGTH  = 256
BATCH_SIZE  = 16
GRAD_ACCUM  = 2       # effective batch = 32
NUM_EPOCHS  = 10
LR          = 2e-5
WEIGHT_DECAY = 0.01

os.environ["WANDB_API_KEY"] = WANDB_API_KEY
print("Config loaded. Labels:", LABEL_NAMES)

In [ ]:
# ── Cell 3: Mount Google Drive (Colab only) ────────────────────────────────
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Uncomment nếu file nằm trên Drive:
    # !cp /content/drive/MyDrive/fshop_data/unified_dataset.csv .
    print("Drive mounted")
else:
    print("Not Colab — skipping Drive mount")

In [ ]:
# ── Cell 4: Load dataset và tính class weights ────────────────────────────
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATASET_PATH)
# Chỉ giữ rows có text hợp lệ
df = df[df["text"].notna() & (df["text"].str.strip() != "")].reset_index(drop=True)
print(f"Total samples: {len(df)}")
print("Label distribution:")
print(df[LABEL_NAMES].sum().to_string())

# Class weights để xử lý imbalance — pos_weight = neg/pos per label
pos_counts = df[LABEL_NAMES].sum().values.astype(float)
neg_counts = len(df) - pos_counts
pos_weights = neg_counts / np.clip(pos_counts, 1, None)
print("\nPos weights (for BCEWithLogitsLoss):")
for lbl, w in zip(LABEL_NAMES, pos_weights):
    print(f"  {lbl}: {w:.2f}")

# Stratified split on 'toxic' (most common label)
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df["toxic"])
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["toxic"])
print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

test_df.to_csv("test_split.csv", index=False)

In [ ]:
# ── Cell 5: Tokenize ──────────────────────────────────────────────────────
import torch
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def df_to_hf_dataset(df):
    ds = Dataset.from_dict({
        "text": df["text"].tolist(),
        **{lbl: df[lbl].tolist() for lbl in LABEL_NAMES}
    })
    def _tokenize(batch):
        enc = tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
        )
        enc["labels"] = [
            [float(batch[lbl][i]) for lbl in LABEL_NAMES]
            for i in range(len(batch["text"]))
        ]
        return enc
    ds = ds.map(_tokenize, batched=True, remove_columns=["text"] + LABEL_NAMES)
    ds.set_format("torch")
    return ds

train_ds = df_to_hf_dataset(train_df)
val_ds   = df_to_hf_dataset(val_df)
print("Tokenization done.")

In [ ]:
# ── Cell 6: Model với weighted loss ───────────────────────────────────────
from sklearn.metrics import f1_score, roc_auc_score
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import wandb

wandb.init(project=WANDB_PROJECT, name=WANDB_RUN_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_NAMES),
    problem_type="multi_label_classification",
)

# Custom Trainer với weighted BCE loss
POS_WEIGHTS = torch.tensor(pos_weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fn = torch.nn.BCEWithLogitsLoss(
            pos_weight=POS_WEIGHTS.to(logits.device)
        )
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    macro_f1     = f1_score(labels, preds, average="macro",  zero_division=0)
    per_label_f1 = f1_score(labels, preds, average=None,     zero_division=0)
    metrics = {"eval_macro_f1": macro_f1}
    for lbl, score in zip(LABEL_NAMES, per_label_f1):
        metrics[f"f1_{lbl}"] = score
    try:
        metrics["eval_roc_auc"] = roc_auc_score(labels, probs, average="macro")
    except Exception:
        pass
    return metrics

print(f"Model: {MODEL_NAME} | Labels: {LABEL_NAMES}")

In [ ]:
# ── Cell 7: Training ──────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",          # 'evaluation_strategy' deprecated
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
    report_to="wandb",
    logging_steps=50,
    run_name=WANDB_RUN_NAME,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# ── Cell 8: Tìm optimal threshold per label ────────────────────────────────
import json

val_preds  = trainer.predict(val_ds)
val_probs  = 1 / (1 + np.exp(-val_preds.predictions))
val_labels = val_preds.label_ids

thresholds = {}
for i, lbl in enumerate(LABEL_NAMES):
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.20, 0.80, 0.05):
        preds_i = (val_probs[:, i] >= t).astype(int)
        f1 = f1_score(val_labels[:, i], preds_i, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    thresholds[lbl] = round(float(best_t), 2)
    print(f"  {lbl}: threshold={best_t:.2f}, best_F1={best_f1:.4f}")

with open("thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=2)
print("thresholds.json saved")

In [ ]:
# ── Cell 9: Push lên HuggingFace Hub ─────────────────────────────────────
from huggingface_hub import login, upload_file

login()  # nhập HF token khi được prompt

trainer.model.push_to_hub(HF_MODEL_REPO)
tokenizer.push_to_hub(HF_MODEL_REPO)
upload_file(
    path_or_fileobj="thresholds.json",
    path_in_repo="thresholds.json",
    repo_id=HF_MODEL_REPO,
)

print(f"✅ Model pushed → https://huggingface.co/{HF_MODEL_REPO}")
print(f"Set trong .env của AI service: PHOBERT_MODERATION_MODEL={HF_MODEL_REPO}")

In [ ]:
# ── Cell 10: Lưu test_split về Drive + kết thúc ───────────────────────────
if IN_COLAB:
    import shutil
    shutil.copy("test_split.csv", "/content/drive/MyDrive/fshop_data/test_split.csv")
    print("test_split.csv → Google Drive")

wandb.finish()
print("Training complete!")